
# Mental Health Late Fusion Model

Notebook ini menggunakan:
- Deep Learning untuk data tabular
- NLP untuk data teks
- Late Fusion Adjustment
- TensorFlow Functional API
- Custom Layer

Tanpa Transformer / IndoBERT.


In [189]:
# =========================================================
# 1. IMPORT LIBRARY
# =========================================================

import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    Input,
    Embedding,
    LSTM,
    Bidirectional,
)

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical
from deep_translator import GoogleTranslator

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.15.0


In [190]:
# =========================================================
# LOAD DATASET ASLI
# =========================================================

tabular_df = pd.read_csv("../data/survey.csv")
text_df = pd.read_csv("../data/mental_heath_unbanlanced.csv")

print("Tabular Shape :", tabular_df.shape)
print("Text Shape    :", text_df.shape)

# =========================================================
# LOAD FULL DATASET
# =========================================================

# gunakan seluruh data asli tanpa sampling
print("\n===== AFTER LOADING DATA =====")

print("Tabular :", tabular_df.shape)
print("Text    :", text_df.shape)

print("\nTabular Preview")
print(tabular_df.head())

print("\nText Preview")
print(text_df.head())

import re

def clean_text(text):

    text = str(text)

    # hapus emoji
    text = re.sub(
        r'[^\x00-\x7F]+',
        ' ',
        text
    )

    # hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# apply ke dataset text
text_df["text"] = text_df["text"].apply(clean_text)

print("Cleaning selesai!")

Tabular Shape : (1259, 27)
Text Shape    : (49612, 3)

===== AFTER LOADING DATA =====
Tabular : (1259, 27)
Text    : (49612, 3)

Tabular Preview
             Timestamp  Age  Gender         Country state self_employed  \
0  2014-08-27 11:29:31   37  Female   United States    IL           NaN   
1  2014-08-27 11:29:37   44       M   United States    IN           NaN   
2  2014-08-27 11:29:44   32    Male          Canada   NaN           NaN   
3  2014-08-27 11:29:46   31    Male  United Kingdom   NaN           NaN   
4  2014-08-27 11:30:22   31    Male   United States    TX           NaN   

  family_history treatment work_interfere    no_employees  ...  \
0             No       Yes          Often            6-25  ...   
1             No        No         Rarely  More than 1000  ...   
2             No        No         Rarely            6-25  ...   
3            Yes       Yes          Often          26-100  ...   
4             No        No          Never         100-500  ...   

       

In [191]:

# =========================================================
# 4. PREPROCESS TABULAR DATA
# =========================================================

target_tab = "treatment"

X_tab = tabular_df.drop(columns=[target_tab])
y_tab = tabular_df[target_tab]

encoders = {}

for col in X_tab.columns:

    le = LabelEncoder()

    X_tab[col] = le.fit_transform(
        X_tab[col].astype(str)
    )

    encoders[col] = le

label_tab = LabelEncoder()

y_tab = label_tab.fit_transform(y_tab)

y_tab_onehot = to_categorical(y_tab)

scaler = StandardScaler()

X_tab = scaler.fit_transform(X_tab)

X_tab_train, X_tab_test, y_tab_train, y_tab_test = train_test_split(
    X_tab,
    y_tab_onehot,
    test_size=0.2,
    random_state=42
)

print(X_tab_train.shape)


(1007, 26)


In [192]:

# =========================================================
# 5. PREPROCESS TEXT DATA
# =========================================================

TEXT_COLUMN = "text"
LABEL_COLUMN = "status"

X_text = text_df[TEXT_COLUMN].astype(str)

label_txt = LabelEncoder()

y_text = label_txt.fit_transform(
    text_df[LABEL_COLUMN]
)

y_txt_onehot = to_categorical(y_text)

X_txt_train, X_txt_test, y_txt_train, y_txt_test = train_test_split(
    X_text,
    y_txt_onehot,
    test_size=0.2,
    random_state=42
)

print(text_df.columns)
print(label_txt.classes_)


Index(['Unique_ID', 'text', 'status'], dtype='object')
['Anxiety' 'Depression' 'Normal' 'Suicidal']


In [193]:

# =========================================================
# 6. CUSTOM LAYER
# =========================================================

class AttentionLayer(layers.Layer):

    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):

        self.W = self.add_weight(
            shape=(input_shape[-1], 1),
            initializer="random_normal",
            trainable=True
        )

    def call(self, inputs):

        score = tf.nn.tanh(
            tf.matmul(inputs, self.W)
        )

        weights = tf.nn.softmax(score, axis=1)

        context = weights * inputs

        context = tf.reduce_sum(
            context,
            axis=1
        )

        return context


In [194]:
# =========================================================
# 7. TABULAR MODEL
# =========================================================

input_tab = layers.Input(shape=(X_tab_train.shape[1],), name="input_tab")

x1 = layers.Dense(128, activation="relu")(input_tab)
x1 = layers.Dropout(0.4)(x1)

x1 = layers.Dense(64, activation="relu")(x1)
x1 = layers.Dropout(0.3)(x1)

x1 = layers.Dense(32, activation="relu")(x1)

out_tab = layers.Dense(2, activation="softmax", name="out_tab")(x1)

model_tab = models.Model(input_tab, out_tab)

model_tab.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

model_tab.summary()

Model: "model_17"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_tab (InputLayer)      [(None, 26)]              0         
                                                                 
 dense_30 (Dense)            (None, 128)               3456      
                                                                 
 dropout_23 (Dropout)        (None, 128)               0         
                                                                 
 dense_31 (Dense)            (None, 64)                8256      
                                                                 
 dropout_24 (Dropout)        (None, 64)                0         
                                                                 
 dense_32 (Dense)            (None, 32)                2080      
                                                                 
 out_tab (Dense)             (None, 2)                 66 

In [195]:
# =========================================================
# EARLY STOPPING & LEARNING RATE SCHEDULER
# =========================================================

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

In [196]:
# augment hanya TRAIN SET

def augment_tabular(X, y, noise_factor=0.001, n_aug=1):
    X = np.array(X)
    y = np.array(y)

    X_aug = [X]
    y_aug = [y]

    for _ in range(n_aug):
        noise = np.random.normal(loc=0.0, scale=noise_factor, size=X.shape)
        X_aug.append(X + noise)
        y_aug.append(y)

    X_aug = np.vstack(X_aug)
    y_aug = np.vstack(y_aug)
    return X_aug, y_aug

X_train_tab_aug, y_train_tab_aug = augment_tabular(
    X_tab_train,
    y_tab_train,
    noise_factor=0.001,
    n_aug=2
)

print("X_train asli :", X_tab_train.shape)
print("X_train aug  :", X_train_tab_aug.shape)


X_train asli : (1007, 26)
X_train aug  : (3021, 26)


In [197]:
# =========================================================
# 8. TRAIN TABULAR MODEL
# =========================================================

history_tab = model_tab.fit(
    X_train_tab_aug,
    y_train_tab_aug,
    validation_data=(X_tab_test, y_tab_test),
    epochs=40,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=2,
)

Epoch 1/40
95/95 - 1s - loss: 0.6529 - accuracy: 0.6207 - val_loss: 0.6040 - val_accuracy: 0.7063 - lr: 0.0010 - 870ms/epoch - 9ms/step
Epoch 2/40
95/95 - 0s - loss: 0.5932 - accuracy: 0.6875 - val_loss: 0.5808 - val_accuracy: 0.7103 - lr: 0.0010 - 201ms/epoch - 2ms/step
Epoch 3/40
95/95 - 0s - loss: 0.5607 - accuracy: 0.7087 - val_loss: 0.5714 - val_accuracy: 0.6905 - lr: 0.0010 - 188ms/epoch - 2ms/step
Epoch 4/40
95/95 - 0s - loss: 0.5405 - accuracy: 0.7282 - val_loss: 0.5534 - val_accuracy: 0.7143 - lr: 0.0010 - 155ms/epoch - 2ms/step
Epoch 5/40
95/95 - 0s - loss: 0.5068 - accuracy: 0.7517 - val_loss: 0.5396 - val_accuracy: 0.7421 - lr: 0.0010 - 187ms/epoch - 2ms/step
Epoch 6/40
95/95 - 0s - loss: 0.4862 - accuracy: 0.7574 - val_loss: 0.5100 - val_accuracy: 0.7540 - lr: 0.0010 - 179ms/epoch - 2ms/step
Epoch 7/40
95/95 - 0s - loss: 0.4450 - accuracy: 0.7858 - val_loss: 0.4847 - val_accuracy: 0.7659 - lr: 0.0010 - 146ms/epoch - 2ms/step
Epoch 8/40
95/95 - 0s - loss: 0.4226 - accuracy:

In [198]:
# =========================================================
# 9. NLP MODEL
# =========================================================

vectorize_layer = layers.TextVectorization(
    max_tokens=10000, output_mode="int", output_sequence_length=100
)

vectorize_layer.adapt(X_txt_train)

input_txt = layers.Input(shape=(1,), dtype=tf.string, name="input_txt")

x2 = vectorize_layer(input_txt)

x2 = layers.Embedding(10000, 128)(x2)

x2 = Bidirectional(LSTM(128, return_sequences=True))(x2)

x2 = AttentionLayer()(x2)

x2 = layers.Dense(64, activation="relu")(x2)

x2 = layers.Dropout(0.4)(x2)

out_txt = layers.Dense(4, activation="softmax", name="out_txt")(x2)

model_txt = models.Model(input_txt, out_txt)

model_txt.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)

model_txt.summary()

Model: "model_18"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_txt (InputLayer)      [(None, 1)]               0         
                                                                 
 text_vectorization_8 (Text  (None, 100)               0         
 Vectorization)                                                  
                                                                 
 embedding_8 (Embedding)     (None, 100, 128)          1280000   
                                                                 
 bidirectional_8 (Bidirecti  (None, 100, 256)          263168    
 onal)                                                           
                                                                 
 attention_layer_16 (Attent  (None, 256)               256       
 ionLayer)                                                       
                                                          

In [199]:
# =========================================================
# 10. TRAIN NLP MODEL
# =========================================================

history_txt = model_txt.fit(
    np.array(X_txt_train).reshape(-1, 1),
    y_txt_train,
    validation_data=(np.array(X_txt_test).reshape(-1, 1), y_txt_test),
    epochs=25,
    batch_size=64,
    callbacks=[early_stopping, reduce_lr],
    verbose=2,
)

Epoch 1/25
621/621 - 159s - loss: 0.7203 - accuracy: 0.6929 - val_loss: 0.5596 - val_accuracy: 0.7656 - lr: 0.0010 - 159s/epoch - 257ms/step
Epoch 2/25
621/621 - 163s - loss: 0.5130 - accuracy: 0.7962 - val_loss: 0.5392 - val_accuracy: 0.7842 - lr: 0.0010 - 163s/epoch - 263ms/step
Epoch 3/25
621/621 - 166s - loss: 0.4445 - accuracy: 0.8256 - val_loss: 0.5548 - val_accuracy: 0.7783 - lr: 0.0010 - 166s/epoch - 267ms/step
Epoch 4/25
621/621 - 170s - loss: 0.3915 - accuracy: 0.8484 - val_loss: 0.5996 - val_accuracy: 0.7646 - lr: 0.0010 - 170s/epoch - 273ms/step
Epoch 5/25
621/621 - 170s - loss: 0.3082 - accuracy: 0.8851 - val_loss: 0.6740 - val_accuracy: 0.7600 - lr: 5.0000e-04 - 170s/epoch - 274ms/step
Epoch 6/25
621/621 - 170s - loss: 0.2682 - accuracy: 0.9013 - val_loss: 0.7267 - val_accuracy: 0.7559 - lr: 5.0000e-04 - 170s/epoch - 274ms/step
Epoch 7/25
Restoring model weights from the end of the best epoch: 2.
621/621 - 173s - loss: 0.2145 - accuracy: 0.9218 - val_loss: 0.8258 - val_ac

In [200]:
# =========================================================
# 11. EVALUATE TABULAR MODEL & NLP MODEL
# =========================================================

loss_tab, acc_tab = model_tab.evaluate(X_tab_test, y_tab_test)
print("Tabular Accuracy:", acc_tab)
print("Tabular Training Accuracy:", max(history_tab.history['accuracy']))

loss_txt, acc_txt = model_txt.evaluate(
    np.array(X_txt_test).reshape(-1,1),
    y_txt_test
)
print("Text Accuracy:", acc_txt)
print("Text Training Accuracy:", max(history_txt.history['accuracy']))

8/8 [==============================] - 0s 2ms/step - loss: 0.4226 - accuracy: 0.7937
Tabular Accuracy: 0.7936508059501648
Tabular Training Accuracy: 0.8831512928009033
311/311 [==============================] - 16s 53ms/step - loss: 0.5392 - accuracy: 0.7842
Text Accuracy: 0.7842386364936829
Text Training Accuracy: 0.9217919111251831


In [203]:
import joblib

joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

joblib.dump(
    encoders,
    "../models/encoders.pkl"
)

joblib.dump(
    label_tab,
    "../models/label_tab.pkl"
)

joblib.dump(
    label_txt,
    "../models/label_txt.pkl"
)

['../models/label_txt.pkl']

In [204]:

# =========================================================
# 17. SAVE MODEL
# =========================================================

model_tab.save("../models/tabular_model.keras")
model_txt.save("../models/text_model.keras")

print("Model berhasil disimpan!")


Model berhasil disimpan!


In [205]:

# =========================================================
# 17. LOAD MODEL
# =========================================================

loaded_tab = tf.keras.models.load_model(
    "../models/tabular_model.keras"
)

loaded_txt = tf.keras.models.load_model(
    "../models/text_model.keras",
    custom_objects={
        "AttentionLayer": AttentionLayer
    }
)

print("Model berhasil di-load kembali!")


Model berhasil di-load kembali!


In [206]:
# =========================================================
# 18. LATE FUSION DYNAMIC
# =========================================================


def translate_to_english(text):
    try:
        translated = GoogleTranslator(source="auto", target="en").translate(text)
        return translated
    except Exception as e:
        print("Translation Error:", e)
        return text


def run_late_fusion_prediction(
    user_text, tabular_dict=None, nlp_weight=0.75, tabular_weight=0.25
):
    # ==========================================
    # TRANSLATION
    # ==========================================

    translated_text = translate_to_english(user_text)
    text_clean = clean_text(translated_text)

    # =====================================================
    # NLP MODEL
    # =====================================================

    prob_txt = loaded_txt.predict(np.array([text_clean]), verbose=0)[0]
    nlp_labels = label_txt.classes_.tolist()
    nlp_probs = dict(zip(nlp_labels, prob_txt))
    primary_nlp_label = nlp_labels[np.argmax(prob_txt)]

    # =====================================================
    # NLP RISK SCORE
    # =====================================================

    risk_keywords = ["suicidal", "depression", "anxiety"]

    nlp_risk = 0

    for label, prob in nlp_probs.items():

        if label.lower() in risk_keywords:
            nlp_risk += prob

    nlp_risk = float(np.clip(nlp_risk, 0, 1))

    # =====================================================
    # TABULAR MODEL
    # =====================================================

    treatment_confidence = None
    tabular_probs = {}

    if tabular_dict is not None:

        row = []

        for col in encoders.keys():

            value = str(tabular_dict.get(col, ""))

            try:
                encoded = encoders[col].transform([value])[0]

            except:
                encoded = 0

            row.append(encoded)

        row = pd.DataFrame([row], columns=list(encoders.keys()))

        row = scaler.transform(row)

        prob_tab = loaded_tab.predict(row, verbose=0)[0]

        tabular_probs = dict(zip(label_tab.classes_, prob_tab))

        treatment_confidence = float(prob_tab[1])

    # =====================================================
    # FUSION
    # =====================================================

    if treatment_confidence is not None:

        fused_risk = nlp_weight * nlp_risk + tabular_weight * treatment_confidence

    else:

        fused_risk = nlp_risk

    fused_risk = float(np.clip(fused_risk, 0, 1))

    # =====================================================
    # DYNAMIC INTERPRETATION
    # =====================================================

    if fused_risk >= 0.85:

        severity = "CRITICAL"

    elif fused_risk >= 0.65:

        severity = "HIGH"

    elif fused_risk >= 0.40:

        severity = "MODERATE"

    else:

        severity = "LOW"

    # =====================================================
    # RECOMMENDATION
    # =====================================================

    recommendations = []

    if fused_risk >= 0.80:

        recommendations.append("Segera konsultasi dengan profesional kesehatan mental.")

    elif fused_risk >= 0.60:

        recommendations.append("Disarankan mencari dukungan profesional.")

    elif fused_risk >= 0.40:

        recommendations.append("Lakukan monitoring kondisi psikologis secara berkala.")

    else:

        recommendations.append("Kondisi relatif stabil.")

    if treatment_confidence is not None:

        if treatment_confidence >= 0.70:

            recommendations.append(
                "Data survei menunjukkan kemungkinan membutuhkan treatment."
            )

        elif treatment_confidence <= 0.30:

            recommendations.append("Data survei menunjukkan risiko treatment rendah.")


    return {
    "original_text": user_text,
    "translated_text": translated_text,
    "primary_label": primary_nlp_label,
    "nlp_probabilities": nlp_probs,
    "nlp_risk": round(nlp_risk, 4),
    "treatment_confidence": treatment_confidence,
    "fused_risk": round(fused_risk, 4),
    "severity": severity,
    "recommendations": recommendations,
}

In [207]:
# =========================================================
# 19. TESTING
# =========================================================

sample_tabular = tabular_df.drop(columns=[target_tab]).iloc[0].to_dict()

examples = [
    "Saya akhir-akhir ini sering merasakan sedih berkepanjangan karena banyak tugas yang menumpuk ditambah tuntutan masa depan",
    "Kepalaku penuh dengan angka-angka yang tidak kunjung menemukan titik temu; rasanya seperti berlari di tempat sementara tagihan terus mengejar",
    "saya banyak cicilan hutang membuat saya frustasi dan tidak ada jalan keluar",
    "Saya merasa bahagia setelah bertemu dengan pacar saya",
]

for text in examples:

    result = run_late_fusion_prediction(text, sample_tabular)

    print("\n")
    print("=" * 70)

    print("TEXT")
    print(text)

    print("\nPRIMARY NLP LABEL")
    print(result["primary_label"])

    print("\nNLP RISK")
    print(result["nlp_risk"])

    print("\nFUSED RISK")
    print(result["fused_risk"])

    print("\nSEVERITY")
    print(result["severity"])

    print("\nRECOMMENDATIONS")

    for rec in result["recommendations"]:

        print("-", rec)

    print("=" * 70)

print("\nORIGINAL TEXT")
print(result["original_text"])

print("\nTRANSLATED TEXT")
print(result["translated_text"])

print("\nPRIMARY NLP LABEL")
print(result["primary_label"])



TEXT
Saya akhir-akhir ini sering merasakan sedih berkepanjangan karena banyak tugas yang menumpuk ditambah tuntutan masa depan

PRIMARY NLP LABEL
Depression

NLP RISK
0.844

FUSED RISK
0.8558

SEVERITY
CRITICAL

RECOMMENDATIONS
- Segera konsultasi dengan profesional kesehatan mental.
- Data survei menunjukkan kemungkinan membutuhkan treatment.


TEXT
Kepalaku penuh dengan angka-angka yang tidak kunjung menemukan titik temu; rasanya seperti berlari di tempat sementara tagihan terus mengejar

PRIMARY NLP LABEL
Normal

NLP RISK
0.1705

FUSED RISK
0.3507

SEVERITY
LOW

RECOMMENDATIONS
- Kondisi relatif stabil.
- Data survei menunjukkan kemungkinan membutuhkan treatment.


TEXT
saya banyak cicilan hutang membuat saya frustasi dan tidak ada jalan keluar

PRIMARY NLP LABEL
Suicidal

NLP RISK
0.8563

FUSED RISK
0.8651

SEVERITY
CRITICAL

RECOMMENDATIONS
- Segera konsultasi dengan profesional kesehatan mental.
- Data survei menunjukkan kemungkinan membutuhkan treatment.


TEXT
Saya merasa bah